# Business Entity Resolution - Data Exploration

## 1. Objective

This notebook performs initial exploratory data analysis for the Amazon ML Challenge Business Entity Resolution problem.

The goal is to understand:

- Dataset dimensions
- Column structure
- Missing values
- Duplicate records
- Business-name variations
- Address variations
- Country distribution
- Ground-truth match distribution
- Singleton entities
- One-to-many matching behaviour

The analysis sections that follow work through these topics one at a time, and each one closes with a short interpretation of the tables it produces.

This notebook does not perform final preprocessing, blocking, model training or prediction.

The observations from this notebook will be used to design the shared preprocessing pipeline used by all three team members.

## 2. Dataset Structure

The challenge provides three independent business-data sources:

- Source 1: deduplicated reference source
- Source 2: matching source
- Source 3: matching source

Training data also contains the ground-truth mapping from Source 1 entities to their matching Source 2 and Source 3 entities.

All files are TSV files and must be read using `sep="\t"`.

In [3]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 160)

In [4]:
PROJECT_ROOT = Path("../..").resolve()

DATA_DIR = PROJECT_ROOT / "student_resource" / "dataset"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

print("Project root:")
print(PROJECT_ROOT)

print("\nTraining directory:")
print(TRAIN_DIR)

print("\nTest directory:")
print(TEST_DIR)

Project root:
E:\ML-Challenge

Training directory:
E:\ML-Challenge\student_resource\dataset\train

Test directory:
E:\ML-Challenge\student_resource\dataset\test


In [5]:
required_train_files = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
]

required_test_files = [
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv",
]

print("TRAIN FILES")
print("-" * 50)

for filename in required_train_files:
    path = TRAIN_DIR / filename
    print(f"{filename:<30} -> {'FOUND' if path.exists() else 'MISSING'}")

print("\nTEST FILES")
print("-" * 50)

for filename in required_test_files:
    path = TEST_DIR / filename
    print(f"{filename:<30} -> {'FOUND' if path.exists() else 'MISSING'}")

TRAIN FILES
--------------------------------------------------
train_source1.tsv              -> FOUND
train_source2.tsv              -> FOUND
train_source3.tsv              -> FOUND
train_ground_truth.tsv         -> FOUND

TEST FILES
--------------------------------------------------
test_source1.tsv               -> FOUND
test_source2.tsv               -> FOUND
test_source3.tsv               -> FOUND


In [6]:
s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

s2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

s3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

ground_truth = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("Training data loaded successfully.")

Training data loaded successfully.


In [7]:
print("Source 1 shape:", s1.shape)
print("Source 2 shape:", s2.shape)
print("Source 3 shape:", s3.shape)
print("Ground truth shape:", ground_truth.shape)

Source 1 shape: (2206821, 4)
Source 2 shape: (5034616, 4)
Source 3 shape: (5285603, 4)
Ground truth shape: (2206821, 2)


## 3. Initial Dataset Size

The following cell reports the number of records available in each source.

Source 1 is the reference population. Source 2 and Source 3 contain potentially matching records.

The size of these datasets is important because exhaustive pairwise comparison would scale poorly. This motivates the need for blocking and candidate generation.

In [8]:
datasets = {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3,
    "Ground Truth": ground_truth,
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print(df.columns.tolist())


Source 1
--------------------------------------------------
['entity_id', 'business_name', 'business_address', 'country']

Source 2
--------------------------------------------------
['entity_id', 'business_name', 'business_address', 'country']

Source 3
--------------------------------------------------
['entity_id', 'business_name', 'business_address', 'country']

Ground Truth
--------------------------------------------------
['source1_entity_id', 'matched_entity_ids']


## 4. Data Types

All four tables were loaded in Cell 6 with `dtype=str` and `keep_default_na=False`, so every column is stored as text and pandas performs no automatic NA coercion.

This is deliberate: entity IDs, country codes and addresses must keep their exact formatting, including leading zeros and punctuation.

The practical consequence is that missing data appears as an empty string rather than `NaN`, which is why section 7 detects missing values explicitly instead of relying on `isna()`.

In [ ]:
dtype_rows = []

for name, df in datasets.items():
    for column in df.columns:
        dtype_rows.append(
            {
                "dataset": name,
                "column": column,
                "dtype": str(df[column].dtype),
                "rows": len(df),
                "non_null": int(df[column].notna().sum()),
                "null": int(df[column].isna().sum()),
            }
        )

dtype_summary = pd.DataFrame(dtype_rows)

display(dtype_summary)

print("\nIn-memory footprint (deep):")
for name, df in datasets.items():
    megabytes = df.memory_usage(index=True, deep=True).sum() / 1e6
    print(f"{name:<15} {megabytes:>10.1f} MB")

## 5. First Few Rows

A quick qualitative look at each table confirms the field layout and shows how unstructured the raw text is.

Source 1 rows are the entities we must predict matches for. Source 2 and Source 3 rows form the candidate pool. The ground truth maps each Source 1 entity to zero or more Source 2 and Source 3 entities.

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 60)
    display(df.head(5))

## 6. Schema Validation

The three source tables are expected to share an identical four-column schema, while the ground truth uses a two-column schema.

Validating the schema up front avoids silent misalignment later, for example when joining ground truth back to Source 1 on the wrong key.

In [ ]:
expected_entity_schema = ["entity_id", "business_name", "business_address", "country"]
expected_ground_truth_schema = ["source1_entity_id", "matched_entity_ids"]

schema_rows = []

for name, df in datasets.items():
    expected = (
        expected_ground_truth_schema if name == "Ground Truth" else expected_entity_schema
    )
    actual = df.columns.tolist()
    schema_rows.append(
        {
            "dataset": name,
            "n_columns": len(actual),
            "column_order_ok": actual == expected,
            "missing_columns": sorted(set(expected) - set(actual)),
            "unexpected_columns": sorted(set(actual) - set(expected)),
        }
    )

schema_summary = pd.DataFrame(schema_rows)

display(schema_summary)

## 7. Missing-Value Analysis

Because the data was read with `keep_default_na=False`, `isna()` reports nothing. Missing content instead appears as an empty string or as whitespace-only text.

Both cases are counted below and kept separate from genuinely populated values. Whitespace-only values matter because they survive naive string comparisons while carrying no information.

In [ ]:
missing_rows = []

for name, df in datasets.items():
    for column in df.columns:
        series = df[column]
        is_empty = series == ""
        is_blank = series.str.strip() == ""
        missing_total = int(is_blank.sum())
        missing_rows.append(
            {
                "dataset": name,
                "column": column,
                "rows": len(series),
                "empty_string": int(is_empty.sum()),
                "whitespace_only": int((is_blank & ~is_empty).sum()),
                "missing_total": missing_total,
                "missing_pct": round(100 * missing_total / len(series), 4),
            }
        )

missing_summary = pd.DataFrame(missing_rows)

display(missing_summary)

## 8. Duplicate Entity IDs

`entity_id` is the primary key within each source, so any duplication would break the join between predictions and the evaluation set.

The ground truth is keyed on `source1_entity_id`, where exactly one row per Source 1 entity is required.

In [ ]:
id_rows = []

for name, df in datasets.items():
    key = "source1_entity_id" if name == "Ground Truth" else "entity_id"
    series = df[key]
    id_rows.append(
        {
            "dataset": name,
            "id_column": key,
            "rows": len(series),
            "distinct_ids": int(series.nunique()),
            "duplicated_ids": int(series.duplicated().sum()),
            "duplicated_pct": round(100 * series.duplicated().mean(), 6),
        }
    )

display(pd.DataFrame(id_rows))

## 9. Duplicate Business Names

Business names repeat heavily across and within sources, because the same real-world business appears in more than one source and a name is not a unique identifier.

Quantifying how weak this field is on its own motivates combining it with address, country and fuzzy similarity signals later.

In [ ]:
name_rows = []

for name, df in datasets.items():
    if "business_name" not in df.columns:
        continue
    series = df["business_name"]
    name_rows.append(
        {
            "dataset": name,
            "rows": len(series),
            "distinct_names": int(series.nunique()),
            "repeated_name_rows": int(series.duplicated().sum()),
            "repeated_name_pct": round(100 * series.duplicated().mean(), 2),
        }
    )

name_summary = pd.DataFrame(name_rows)

display(name_summary)

print("\nMost frequent business names in Source 1:")
display(
    s1["business_name"]
    .value_counts()
    .head(15)
    .rename_axis("business_name")
    .reset_index(name="rows")
    .assign(pct=lambda frame: (100 * frame["rows"] / len(s1)).round(2))
)

## 10. Duplicate Addresses

Addresses repeat for the same reason as names and are usually noisier, with abbreviations, missing house numbers and inconsistent spacing.

A case-folded and whitespace-normalised version is compared alongside the raw field, which separates genuine repetition from formatting differences.

In [ ]:
address_rows = []

for name, df in datasets.items():
    if "business_address" not in df.columns:
        continue
    series = df["business_address"]
    normalised = series.str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
    address_rows.append(
        {
            "dataset": name,
            "rows": len(series),
            "distinct_addresses": int(series.nunique()),
            "repeated_raw_pct": round(100 * series.duplicated().mean(), 2),
            "distinct_normalised": int(normalised.nunique()),
            "repeated_normalised_pct": round(100 * normalised.duplicated().mean(), 2),
        }
    )

address_summary = pd.DataFrame(address_rows)

display(address_summary)

print("\nMost frequent addresses in Source 1:")
display(
    s1["business_address"]
    .value_counts()
    .head(15)
    .rename_axis("business_address")
    .reset_index(name="rows")
    .assign(pct=lambda frame: (100 * frame["rows"] / len(s1)).round(2))
)

## 11. Country Distribution

`country` is treated as an open-set label. No country is assumed to be present or dominant, and the values are only inspected, never hard-coded.

Country is a strong disambiguation signal when comparing candidates, but only across the set of values that actually occur in the data.

In [ ]:
country_rows = []

for name, df in datasets.items():
    if "country" not in df.columns:
        continue
    counts = df["country"].value_counts()
    country_rows.append(
        {
            "dataset": name,
            "rows": len(df),
            "distinct_values": int(counts.size),
            "blank_rows": int((df["country"].str.strip() == "").sum()),
            "top_value": counts.index[0],
            "top_rows": int(counts.iloc[0]),
            "top_pct": round(100 * counts.iloc[0] / len(df), 2),
            "top5_pct": round(100 * counts.head(5).sum() / len(df), 2),
        }
    )

display(pd.DataFrame(country_rows))

for name, df in datasets.items():
    if "country" not in df.columns:
        continue
    print(f"\nTop 15 country values in {name}")
    print("-" * 60)
    display(
        df["country"]
        .value_counts()
        .head(15)
        .rename_axis("country")
        .reset_index(name="rows")
        .assign(pct=lambda frame: (100 * frame["rows"] / len(df)).round(2))
    )

## 12. Business-Name Exploration

Name length, token count and formatting habits indicate how noisy a field is and which normalisations are worth applying.

Legal-form suffixes are counted as a generic international pattern rather than as country knowledge, since comparable abbreviation styles appear across many registries.

In [ ]:
name_profile_rows = []

for name, df in datasets.items():
    if "business_name" not in df.columns:
        continue
    series = df["business_name"]
    name_profile_rows.append(
        {
            "dataset": name,
            "rows": len(series),
            "char_mean": round(series.str.len().mean(), 1),
            "char_median": series.str.len().median(),
            "char_p95": series.str.len().quantile(0.95),
            "tokens_mean": round(series.str.count(r"\s+").add(1).mean(), 2),
            "tokens_median": series.str.count(r"\s+").add(1).median(),
            "has_digit_pct": round(100 * series.str.contains(r"\d", regex=True).mean(), 2),
            "is_uppercase_pct": round(100 * series.str.isupper().mean(), 2),
            "has_punctuation_pct": round(
                100 * series.str.contains(r"[^\w\s]", regex=True).mean(), 2
            ),
        }
    )

display(pd.DataFrame(name_profile_rows))

In [ ]:
LEGAL_SUFFIXES = [
    "LLC", "INC", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY",
    "GMBH", "AG", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV",
    "AB", "AS", "OY", "PTE", "PTY", "PLC", "KG", "KK", "BHD", "PVT",
    "PRIVATE", "HOLDING", "GROUP", "TRADING", "ENTERPRISES",
]

legal_suffix_pattern = r"\b(?:" + "|".join(LEGAL_SUFFIXES) + r")\b"

legal_suffix_rows = []

for name, df in datasets.items():
    if "business_name" not in df.columns:
        continue
    upper_names = df["business_name"].str.upper()
    legal_suffix_rows.append(
        {
            "dataset": name,
            "rows": len(df),
            "with_legal_suffix_pct": round(
                100 * upper_names.str.contains(legal_suffix_pattern, regex=True).mean(), 2
            ),
        }
    )

display(pd.DataFrame(legal_suffix_rows))

print("\nMost common legal-form tokens in Source 1:")
display(
    s1["business_name"]
    .str.upper()
    .str.findall(legal_suffix_pattern)
    .explode()
    .value_counts()
    .head(15)
    .rename_axis("suffix")
    .reset_index(name="occurrences")
)

## 13. Address Exploration

Addresses carry the most source-specific signal, so their structure is profiled: length, token count, numeric content and the use of street-type abbreviations.

Abbreviation tokens are treated as generic patterns rather than being tied to any single country's addressing convention.

In [ ]:
STREET_TOKENS = [
    "ST", "STR", "STREET", "AVE", "AVENUE", "RD", "ROAD", "BLVD", "BOULEVARD",
    "DR", "DRIVE", "LN", "LANE", "PL", "PLACE", "SQ", "SQUARE", "CT", "COURT",
    "SUITE", "STE", "UNIT", "FLOOR", "FL", "NO", "NR", "PLZ", "HAUS",
]

street_token_pattern = r"\b(?:" + "|".join(STREET_TOKENS) + r")\b"

address_profile_rows = []

for name, df in datasets.items():
    if "business_address" not in df.columns:
        continue
    series = df["business_address"]
    address_profile_rows.append(
        {
            "dataset": name,
            "rows": len(series),
            "char_mean": round(series.str.len().mean(), 1),
            "char_median": series.str.len().median(),
            "char_p95": series.str.len().quantile(0.95),
            "tokens_mean": round(series.str.count(r"\s+").add(1).mean(), 2),
            "has_digit_pct": round(100 * series.str.contains(r"\d", regex=True).mean(), 2),
            "has_comma_pct": round(100 * series.str.contains(",", regex=False).mean(), 2),
            "has_street_token_pct": round(
                100 * series.str.upper().str.contains(street_token_pattern, regex=True).mean(), 2
            ),
        }
    )

display(pd.DataFrame(address_profile_rows))

## 14. Entity ID and Source-Prefix Validation

There is no dedicated source column. A record's source is encoded in its `entity_id` prefix (`S1-`, `S2-`, `S3-`), so each prefix is validated against the file the record came from.

Any mismatch would mean a row is mislabelled and would silently corrupt candidate generation. The prefixes inside `matched_entity_ids` are validated separately in section 19.

In [ ]:
EXPECTED_PREFIX = {
    "Source 1": "S1-",
    "Source 2": "S2-",
    "Source 3": "S3-",
    "Ground Truth": "S1-",
}

entity_id_prefix_pattern = r"^S[123]-"

prefix_rows = []

for name, df in datasets.items():
    key = "source1_entity_id" if name == "Ground Truth" else "entity_id"
    series = df[key]
    expected = EXPECTED_PREFIX[name]
    prefix_rows.append(
        {
            "dataset": name,
            "id_column": key,
            "expected_prefix": expected,
            "rows": len(series),
            "well_formed": int(series.str.match(entity_id_prefix_pattern, na=False).sum()),
            "wrong_source_prefix": int((~series.str.startswith(expected, na=False)).sum()),
            "distinct_prefixes": series.str.extract(r"^(S[123])-", expand=False).nunique(),
        }
    )

display(pd.DataFrame(prefix_rows))

print("\nSample identifiers per dataset:")
for name, df in datasets.items():
    key = "source1_entity_id" if name == "Ground Truth" else "entity_id"
    print(f"{name:<15} {df[key].head(3).tolist()}")

## 15. Ground-Truth Analysis

The ground truth must contain exactly one row per Source 1 entity, otherwise the per-entity macro score cannot be computed.

Coverage is checked in both directions: Source 1 entities absent from the ground truth, and ground-truth keys that do not exist in Source 1.

In [ ]:
source1_id_index = pd.Index(s1["entity_id"])
ground_truth_id_index = pd.Index(ground_truth["source1_entity_id"])

ground_truth_overview = {
    "rows": len(ground_truth),
    "distinct_source1_ids": int(ground_truth["source1_entity_id"].nunique()),
    "duplicated_source1_rows": int(ground_truth["source1_entity_id"].duplicated().sum()),
    "source1_entities": len(source1_id_index),
    "source1_ids_covered": int(source1_id_index.isin(ground_truth_id_index).sum()),
    "source1_ids_not_covered": int((~source1_id_index.isin(ground_truth_id_index)).sum()),
    "ground_truth_ids_not_in_source1": int((~ground_truth_id_index.isin(source1_id_index)).sum()),
    "rows_with_no_matches": int((ground_truth["matched_entity_ids"].str.strip() == "").sum()),
}

display(pd.Series(ground_truth_overview, name="value").to_frame())

## 16. Match-Count Distribution

`matched_entity_ids` is a comma-separated list, so the number of matches per entity is the comma count plus one, and zero when the list is empty.

The shape of this distribution drives the whole modelling strategy under a macro F0.5 score, because every Source 1 entity contributes equally to the average.

In [ ]:
raw_match_lists = ground_truth["matched_entity_ids"]
has_matches = raw_match_lists.str.strip() != ""

match_count = raw_match_lists.str.count(",").add(1).where(has_matches, 0).astype("int64")

match_count_distribution = (
    match_count.value_counts()
    .sort_index()
    .rename_axis("match_count")
    .reset_index(name="entities")
)

match_count_distribution["pct_of_source1"] = (
    100 * match_count_distribution["entities"] / len(ground_truth)
).round(2)

display(match_count_distribution)

print("\nSummary statistics:")
display(match_count.describe().to_frame("value"))

In [ ]:
group_labels = np.select(
    [match_count == 0, match_count == 1, match_count >= 2],
    ["no match (singleton)", "exactly 1 match", "2 or more matches"],
    default="other",
)

match_groups = pd.DataFrame(
    {
        "source1_entity_id": ground_truth["source1_entity_id"],
        "group": group_labels,
        "links": match_count,
    }
)

group_summary = (
    match_groups.groupby("group", observed=True)
    .agg(entities=("links", "size"), links=("links", "sum"))
    .reindex(["no match (singleton)", "exactly 1 match", "2 or more matches"])
    .fillna(0)
    .astype({"entities": "int64", "links": "int64"})
    .reset_index()
)

group_summary["pct_of_source1"] = (100 * group_summary["entities"] / len(ground_truth)).round(2)
group_summary["pct_of_all_links"] = (
    100 * group_summary["links"] / max(int(match_count.sum()), 1)
).round(2)

display(group_summary)

## 17. Singleton Analysis

A singleton is a Source 1 entity with no true match. Under macro F0.5 every entity is weighted equally, so singletons contribute as much to the final score as heavily matched entities.

Predicting an empty list for a true singleton scores 1.0, while predicting anything for it scores 0.0. Field completeness is compared below to see whether singletons differ systematically from matched entities.

In [ ]:
match_count_by_id = pd.Series(
    match_count.to_numpy(),
    index=pd.Index(ground_truth["source1_entity_id"], name="source1_entity_id"),
    name="match_count",
)

source1_profiled = s1.assign(
    match_count=match_count_by_id.reindex(s1["entity_id"]).to_numpy()
)

singleton_mask = source1_profiled["match_count"] == 0

field_completeness = []

for label, mask in [
    ("singleton (0 matches)", singleton_mask),
    ("matched (1 or more)", ~singleton_mask),
]:
    subset = source1_profiled.loc[mask]
    field_completeness.append(
        {
            "group": label,
            "entities": len(subset),
            "blank_name_pct": round(100 * (subset["business_name"].str.strip() == "").mean(), 2),
            "blank_address_pct": round(
                100 * (subset["business_address"].str.strip() == "").mean(), 2
            ),
            "blank_country_pct": round(100 * (subset["country"].str.strip() == "").mean(), 2),
            "mean_name_chars": round(subset["business_name"].str.len().mean(), 1),
            "mean_address_chars": round(subset["business_address"].str.len().mean(), 1),
        }
    )

display(pd.DataFrame(field_completeness))

print(f"\nSingletons: {int(singleton_mask.sum()):,} of {len(source1_profiled):,} Source 1 entities")
print("\nExample singletons:")
display(
    source1_profiled.loc[
        singleton_mask, ["entity_id", "business_name", "business_address", "country"]
    ].head(10)
)

## 18. Multi-Match Analysis

One-to-many cases are common: a single Source 1 entity can match several records in Source 2 and Source 3.

Two consequences follow. Candidate generation must be able to return more than one candidate per entity, and because F0.5 weights precision more heavily than recall, each extra incorrect candidate costs more than a single missed one.

In [ ]:
multi_match_mask = source1_profiled["match_count"] >= 2

multi_match_distribution = (
    match_count[match_count >= 2]
    .value_counts()
    .sort_index()
    .rename_axis("match_count")
    .reset_index(name="entities")
)

multi_match_distribution["pct_of_source1"] = (
    100 * multi_match_distribution["entities"] / len(source1_profiled)
).round(2)

display(multi_match_distribution)

total_links = int(match_count.sum())

print(f"\nEntities with 2 or more matches: {int(multi_match_mask.sum()):,}")
print(
    "Share of all ground-truth links held by those entities: "
    f"{100 * int(source1_profiled.loc[multi_match_mask, 'match_count'].sum()) / total_links:.2f}%"
)
print(f"Maximum matches for a single Source 1 entity: {int(match_count.max())}")

print("\nHighest fan-out entities:")
display(
    source1_profiled.nlargest(10, "match_count")[
        ["entity_id", "business_name", "business_address", "country", "match_count"]
    ]
)

## 19. Ground-Truth ID Validation

Every identifier inside `matched_entity_ids` is expanded and checked against Source 2 and Source 3.

Four properties are verified: each link resolves to a real record, no link points back at Source 1, the same identifier is not repeated within a single row, and the expanded link total agrees with the match counts from section 16.

In [ ]:
ground_truth_links = (
    ground_truth[["source1_entity_id", "matched_entity_ids"]]
    .assign(matched_entity_ids=lambda frame: frame["matched_entity_ids"].str.split(","))
    .explode("matched_entity_ids")
)

ground_truth_links = ground_truth_links[
    ground_truth_links["matched_entity_ids"].str.strip() != ""
]

source2_id_index = pd.Index(s2["entity_id"])
source3_id_index = pd.Index(s3["entity_id"])

resolved_in_source2 = ground_truth_links["matched_entity_ids"].isin(source2_id_index)
resolved_in_source3 = ground_truth_links["matched_entity_ids"].isin(source3_id_index)

link_validation = {
    "expanded_links": len(ground_truth_links),
    "sum_of_match_counts": int(match_count.sum()),
    "resolved_in_source2": int(resolved_in_source2.sum()),
    "resolved_in_source3": int(resolved_in_source3.sum()),
    "unresolved_links": int((~resolved_in_source2 & ~resolved_in_source3).sum()),
    "links_pointing_to_source1": int(
        ground_truth_links["matched_entity_ids"].str.startswith("S1-", na=False).sum()
    ),
    "repeated_id_within_row": int(
        ground_truth_links.duplicated(
            subset=["source1_entity_id", "matched_entity_ids"]
        ).sum()
    ),
    "shared_id_across_source1_rows": int(
        ground_truth_links["matched_entity_ids"].duplicated().sum()
    ),
}

display(pd.Series(link_validation, name="value").to_frame())

del ground_truth_links

## 20. Source 2 and Source 3 Match Composition

Because matches may come from Source 2, Source 3 or both, the composition is measured per Source 1 entity.

This shows whether a single-source approach would forfeit recall, and it cross-checks the counting method used in section 16.

In [ ]:
source2_links_per_entity = raw_match_lists.str.count("S2-")
source3_links_per_entity = raw_match_lists.str.count("S3-")
total_links_per_entity = source2_links_per_entity + source3_links_per_entity

has_source2 = source2_links_per_entity > 0
has_source3 = source3_links_per_entity > 0

source2_only = has_source2 & ~has_source3
source3_only = ~has_source2 & has_source3
both_sources = has_source2 & has_source3

composition = pd.DataFrame(
    [
        {
            "composition": "singleton (no links)",
            "entities": int((~has_source2 & ~has_source3).sum()),
            "links": 0,
        },
        {
            "composition": "Source 2 only",
            "entities": int(source2_only.sum()),
            "links": int(total_links_per_entity[source2_only].sum()),
        },
        {
            "composition": "Source 3 only",
            "entities": int(source3_only.sum()),
            "links": int(total_links_per_entity[source3_only].sum()),
        },
        {
            "composition": "Source 2 and Source 3",
            "entities": int(both_sources.sum()),
            "links": int(total_links_per_entity[both_sources].sum()),
        },
    ]
)

composition["pct_of_source1"] = (100 * composition["entities"] / len(ground_truth)).round(2)

display(composition)

print(f"\nTotal links to Source 2: {int(source2_links_per_entity.sum()):,}")
print(f"Total links to Source 3: {int(source3_links_per_entity.sum()):,}")
print(f"Source 1 entities with at least one Source 2 link: {int(has_source2.sum()):,}")
print(f"Source 1 entities with at least one Source 3 link: {int(has_source3.sum()):,}")
print(
    "Counting cross-check, entities where the prefix total differs from match_count: "
    f"{int((total_links_per_entity != match_count).sum()):,}"
)

## 21. Final Training-Data Summary

The table below consolidates the measurements that will constrain the shared preprocessing pipeline.

In [ ]:
final_summary = pd.DataFrame(
    [
        {"metric": "Source 1 records", "value": len(s1)},
        {"metric": "Source 2 records", "value": len(s2)},
        {"metric": "Source 3 records", "value": len(s3)},
        {"metric": "Ground-truth rows", "value": len(ground_truth)},
        {"metric": "Total ground-truth links", "value": total_links},
        {"metric": "Mean links per Source 1 entity", "value": round(match_count.mean(), 4)},
        {"metric": "Maximum links for one entity", "value": int(match_count.max())},
        {
            "metric": "Singleton share of Source 1 (%)",
            "value": round(100 * singleton_mask.mean(), 2),
        },
        {
            "metric": "Share with 2 or more matches (%)",
            "value": round(100 * multi_match_mask.mean(), 2),
        },
        {"metric": "Source 1 entities with a Source 2 link", "value": int(has_source2.sum())},
        {"metric": "Source 1 entities with a Source 3 link", "value": int(has_source3.sum())},
        {"metric": "Distinct country values in Source 1", "value": int(s1["country"].nunique())},
        {
            "metric": "Source 1 rows with repeated names (%)",
            "value": round(100 * s1["business_name"].duplicated().mean(), 2),
        },
        {
            "metric": "Source 1 rows with repeated addresses (%)",
            "value": round(100 * s1["business_address"].duplicated().mean(), 2),
        },
        {
            "metric": "Source 1 rows with a blank address (%)",
            "value": round(100 * (s1["business_address"].str.strip() == "").mean(), 2),
        },
    ]
)

display(final_summary)

### Key Observations

- All three sources share one four-column text schema and `entity_id` is unique within each file, so those keys can be trusted for joins.
- The data is read as text with no automatic NA handling, so missing content appears as empty or whitespace-only strings and must be detected explicitly.
- Names and addresses repeat heavily and are nowhere near unique, confirming that no single field can resolve entities on its own.
- Country behaves as an open-set label with a long tail, so it is used for comparison and filtering rather than being enumerated in code.
- Entity ID prefixes reliably encode the source, which makes them a cheap validation check and a natural first blocking key.
- The ground truth covers every Source 1 entity exactly once, and each identifier in it resolves to a real Source 2 or Source 3 record.
- A large share of Source 1 entities are singletons and a further group has two or more matches. Under macro F0.5 both carry equal per-entity weight, and precision is weighted more heavily than recall.
- Matches are drawn from both Source 2 and Source 3, so a single-source approach would forfeit recall on part of the population.

## Next Step

These observations are the design constraints for the shared `src/preprocessing.py`, which every team member will import rather than reimplement.

The module will own a single canonical loader for the four training files and three test files, using the text-preserving read options confirmed in section 4 so that no member loses leading zeros or exact formatting.

It will also own the normalisation rules implied by sections 7, 9, 10, 12 and 13: explicit empty and whitespace-only detection, case folding, whitespace collapsing and token-level cleanup of names and addresses. All of this is applied to derived columns so the raw frames stay untouched.

Country is kept as an open-set string with only case and whitespace normalisation, never mapped to a hard-coded list of values. The source-prefix convention from section 14 is exposed as a validation helper, and the match-list parsing from sections 16, 19 and 20 lives in one place so the counting logic stays consistent across the project.

The exploratory tables built here stay in this notebook and are deliberately not moved into the library. Blocking, feature construction, validation splitting, model training and prediction are added in later stages, once this shared foundation is agreed.